In [0]:
-- Anüpassung an databricks Claude 08.09.2026
-- wh_04_Projekt Sprint 4 - Views: BI-ready Output im NorthRiver Warehouse
-- =========================================================================
-- Angepasst für Databricks (workspace.northriver_warehouse), Original war
-- PostgreSQL/Beekeeper. Syntax ist größtenteils identisch (Standard-SQL),
-- angepasst wurden: CREATE OR REPLACE VIEW, drei-Teil-Namen,
-- fact_orders -> fact_order_items, order_item_sk -> order_item_id.
-- =========================================================================

-- ===========================================================================
-- Übung 1 — Eine denormalisierte, BI-ready Order View
-- ===========================================================================

CREATE OR REPLACE VIEW workspace.northriver_warehouse.vw_orders_bi AS
SELECT fo.order_item_id, fo.order_id,
       dc.region, dc.segment,        -- readable customer attributes
       dp.category,                  -- readable product attribute
       ch.channel_name,              -- readable channel label, not channel_sk
       dd.year, dd.month,            -- calendar labels from the date dimension
       fo.quantity, fo.unit_price, fo.revenue
FROM workspace.northriver_warehouse.fact_order_items fo
JOIN workspace.northriver_warehouse.dim_customer dc ON fo.customer_sk = dc.customer_sk
JOIN workspace.northriver_warehouse.dim_product  dp ON fo.product_sk  = dp.product_sk
JOIN workspace.northriver_warehouse.dim_channel  ch ON fo.channel_sk  = ch.channel_sk
JOIN workspace.northriver_warehouse.dim_date     dd ON fo.date_sk     = dd.date_sk;

-- alle Rows anzeigen (erwartet: 32 Rows, je eine pro Order Item):
SELECT * FROM workspace.northriver_warehouse.vw_orders_bi;

-- ===========================================================================
-- Übung 2 — Eine voraggregierte Summary View
-- ===========================================================================

CREATE OR REPLACE VIEW workspace.northriver_warehouse.vw_revenue_by_region_month AS
SELECT region, year, month,
       SUM(revenue)              AS total_revenue,
       COUNT(DISTINCT order_id)  AS orders   -- distinct orders, not order lines
FROM workspace.northriver_warehouse.vw_orders_bi
GROUP BY region, year, month;

SELECT * FROM workspace.northriver_warehouse.vw_revenue_by_region_month;

-- ===========================================================================
-- Übung 3 — Eine Cross-tab (Pivot) für ein kompaktes Grid
-- ===========================================================================
-- Hinweis: Kategorien an dein echtes dim_product angepasst
-- (Footwear/Outerwear/Equipment/Clothing/Accessories statt
-- Furniture/Lighting/Electronics/Office Supplies aus dem Postgres-Original).
-- Erst prüfen mit: SELECT DISTINCT category FROM workspace.northriver_warehouse.dim_product;

CREATE OR REPLACE VIEW workspace.northriver_warehouse.vw_region_category_pivot AS
SELECT dc.region,
       SUM(CASE WHEN dp.category='Footwear'    THEN fo.revenue ELSE 0 END) AS footwear,
       SUM(CASE WHEN dp.category='Outerwear'   THEN fo.revenue ELSE 0 END) AS outerwear,
       SUM(CASE WHEN dp.category='Equipment'   THEN fo.revenue ELSE 0 END) AS equipment,
       SUM(CASE WHEN dp.category='Clothing'    THEN fo.revenue ELSE 0 END) AS clothing,
       SUM(CASE WHEN dp.category='Accessories' THEN fo.revenue ELSE 0 END) AS accessories
FROM workspace.northriver_warehouse.fact_order_items fo
JOIN workspace.northriver_warehouse.dim_customer dc ON fo.customer_sk = dc.customer_sk
JOIN workspace.northriver_warehouse.dim_product  dp ON fo.product_sk  = dp.product_sk
GROUP BY dc.region;

SELECT * FROM workspace.northriver_warehouse.vw_region_category_pivot;

-- ===========================================================================
-- Übung 4 — Unpivot mit UNION ALL
-- ===========================================================================

CREATE OR REPLACE VIEW workspace.northriver_warehouse.vw_channel_metrics_long AS
SELECT channel_name, 'total_revenue'  AS metric, SUM(revenue)            AS value
FROM workspace.northriver_warehouse.vw_orders_bi GROUP BY channel_name
UNION ALL
SELECT channel_name, 'total_quantity', SUM(quantity)
FROM workspace.northriver_warehouse.vw_orders_bi GROUP BY channel_name
UNION ALL
SELECT channel_name, 'order_count',    COUNT(DISTINCT order_id)
FROM workspace.northriver_warehouse.vw_orders_bi GROUP BY channel_name;

-- erwartet: 3 Channels x 3 Metrics = 9 Rows (statt 12, da dein Projekt nur
-- 3 Channels hat: Online/Wholesale/Retail, nicht 4)
SELECT * FROM workspace.northriver_warehouse.vw_channel_metrics_long
ORDER BY channel_name, metric;

-- ===========================================================================
-- Übung 5 — Diese View lesen und ihren echten Grain benennen
-- ===========================================================================

CREATE OR REPLACE VIEW workspace.northriver_warehouse.vw_orders_broken AS
SELECT fo.order_id, dc.region, fo.revenue
FROM workspace.northriver_warehouse.fact_order_items fo
JOIN workspace.northriver_warehouse.dim_customer dc ON fo.customer_sk = dc.customer_sk;

SELECT * FROM workspace.northriver_warehouse.vw_orders_broken;

-- Beweis für den Grain (bei dir: 32 Lines, 29 echte Orders statt 180/178):
SELECT COUNT(*) AS rows_in_view,           -- order lines
       COUNT(DISTINCT order_id) AS orders  -- actual orders
FROM workspace.northriver_warehouse.vw_orders_broken;
